# 4.2.1.1 Створити програму обчислення суми ряду.

$\sum_{i} a_i \cdot \sin{x_i}$, де $i=0,1,…,N$; $N=240 000$, $x_i = 0.0001\cdot i$, $a_i$ – випадкові дійсні числа в діапазоні (-1, 1). Функцію $\sin{x_i}$ обчислювати за допомогою розкладання її в ряд Тейлора: $\sin{x}=x-\frac{x^3}{3!}+\frac{x^5}{5!}-...+(-1)^n \frac{x^{2n+1}}{(2 \cdot n + 1)!}+...$

Обчислення кожного значення функції $sin(x_i)$ можна організувати за наступним алгоритмом:
1. y = x; s = y; k = 1;
2. y = $-\frac{x^2}{(k+1)\cdot (k+2)}\cdot y$
3. s = s + y; k = k + 2;
4. якщо $k \le K$, то перехід на п. б), інакше — кінець.

Для обчислення синуса в програмі створити окрему функцію, K нехай дорівнює 500.

In [1]:
from mpi4py import MPI
import random
import numpy as np

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

def custom_sin(x, this_K=500):
    y = x
    s = y
    k = 1
    while k <= this_K:
        y = -((x**2) / ((k + 1) * (k + 2))) * y
        s = s + y
        k = k + 2
    return s

N = 240_000
K = 500
# How many elements does one process handle
chunk_size = N // size

# Data preparation on main process
a_full = None
x_full = None
if rank == 0:
    a_full = np.array([random.uniform(-1, 1) for _ in range(N)], dtype="d")
    x_full = np.array([0.0001 * i for i in range(N)], dtype="d")

# Initializing buffers to accept data
a_local = np.empty(chunk_size, dtype="d")
x_local = np.empty(chunk_size, dtype="d")

# 4.2.1.3 Timing the Scatter function
t_scatter_start = MPI.Wtime()
comm.Scatter(a_full, a_local, root=0)
comm.Scatter(x_full, x_local, root=0)
t_scatter_end = MPI.Wtime()

# Starting the timer
start_time = MPI.Wtime()

local_sum = 0.0
for i in range(chunk_size):
    local_sum += a_local[i] * custom_sin(x_local[i], this_K=K)

# Collecting results from all processes
total_sum = comm.reduce(local_sum, op=MPI.SUM, root=0)

end_time = MPI.Wtime()

if rank == 0:
    print(f"Proceses amount: {size}")
    print(f"Total sum: {total_sum}")
    print(f"Running time (Tp): {end_time - start_time:.4f} sec")
    print(f"Scatter time: {t_scatter_end - t_scatter_start:.6f} sec")


Proceses amount: 1
Total sum: -44.749436265304396
Running time (Tp): 24.8317 sec
Scatter time: 0.001168 sec
